# CIFAR-10 CLAQ Experiment

A notebook for the CIFAR-10 CLAQ workflow. Baseline and CLAQ policies are trained over five random seeds; qualitative replay examples use the first seed only.

- load config, CLIP, concepts, and data
- load or train Concept-QA
- train baseline and CLAQ policies with a conditional sensitive predictor, `P(S | knowledge state, Y)`
- probe the sensitive concept set before retraining
- check sample-level sensitive-label sanity
- replay tiny-start cases


In [1]:
%load_ext autoreload
%autoreload 2
import json
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from claq.analysis import (
    plot_rollout_comparisons,
    sample_intuition_replays,
)
from claq.config import Cifar10ClaqConfig, default_paths
from claq.core import (
    build_concept_dictionary,
    concept_answers_batch,
    file_sha256,
    load_answer_cache,
    load_clip_model,
    load_concept_qa_checkpoint,
    load_concepts,
    load_run_bundle,
    make_cached_answer_loader,
    make_sensitive_mask,
    save_answer_cache,
    save_bundle_checkpoint,
)
from claq.data import get_cifar10_datasets, get_cifar10_loaders, get_raw_cifar10_dataset
from claq.sensitive_labels import (
    build_cifar10_sensitive_match,
    build_sensitive_labels,
    load_sensitive_labels,
    save_sensitive_labels,
)
from claq.training import HistorySamplingConfig, build_claq_models, fit_claq, seed_everything

In [2]:
repo_root = Path.cwd().resolve()
if not (repo_root / "claq").exists() and (repo_root.parent / "claq").exists():
    repo_root = repo_root.parent

paths = default_paths(repo_root=repo_root)
paths.ensure_artifact_dirs()

config = Cifar10ClaqConfig()
device = config.device
SEEDS = (0, 1, 2, 3, 4)
QUALITATIVE_SEED = SEEDS[0]
seed_everything(QUALITATIVE_SEED)

# The versioned name prevents loading checkpoints trained with the former
# unconditional sensitive predictor P(S | knowledge state).
experiment_name = "cifar10_conditional_y"
FORCE_REBUILD_ANSWER_CACHE = False

# Change this to ".pdf" or ".png" when needed.
figure_ext = ".svg"


def figure_path(stem):
    return paths.figures_root / f"{stem}{figure_ext}"


print(f"repo_root: {repo_root}")
print(f"artifacts_root: {paths.artifacts_root}")
print(f"device: {device}")
print(f"figure_ext: {figure_ext}")

repo_root: /home/jupyter/claq
artifacts_root: /home/jupyter/claq/artifacts
device: cuda
figure_ext: .svg


In [3]:
model_clip, preprocess = load_clip_model(config.clip_model_name, device=device)
concepts = load_concepts(paths.concept_file)
dictionary = build_concept_dictionary(model_clip=model_clip, concepts=concepts, device=device)
sensitive_match = build_cifar10_sensitive_match(concepts)
sens_idx = sensitive_match.indices
sensitive_mask = make_sensitive_mask(config.max_queries, sens_idx, device)

train_ds, test_ds = get_cifar10_datasets(transform=preprocess, root=paths.data_root)
train_loader, test_loader = get_cifar10_loaders(
    transform=preprocess,
    root=paths.data_root,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
)
raw_test_ds = get_raw_cifar10_dataset(paths.data_root, train=False)

print(f"# concepts: {len(concepts)}")
print(f"# sensitive concepts matched: {len(sensitive_match.matched)}")
print(sensitive_match.matched)
if sensitive_match.missing:
    print(f"# sensitive concepts missing: {len(sensitive_match.missing)}")
    print(sensitive_match.missing)

# concepts: 128
# sensitive concepts matched: 23
['a bridle', 'a cab for the driver', 'a captain', 'a collar', 'a copilot', 'a dashboard', 'a driver', 'a flight attendant', 'a gear shift', 'a halter', 'a hitch', 'a lead rope', 'a leash', 'a passenger', 'a pedal', 'a pilot', 'a reins', 'a rider', 'a rifle', 'a saddle', 'a seatbelt', 'a steering wheel', 'a trailer']


In [4]:
qa_checkpoint = paths.checkpoints_root / "concept_qa_cifar10.pt"
qa_source = qa_checkpoint
if qa_checkpoint.exists():
    answering_model = load_concept_qa_checkpoint(qa_checkpoint, device=device)
elif paths.bootstrap_concept_qa_checkpoint.exists():
    answering_model = load_concept_qa_checkpoint(paths.bootstrap_concept_qa_checkpoint, device=device)
    qa_source = paths.bootstrap_concept_qa_checkpoint
else:
    raise FileNotFoundError("No local or bootstrap Concept-QA checkpoint available for CIFAR-10.")

print(f"Concept-QA ready from: {qa_source}")

Concept-QA ready from: /home/jupyter/claq/artifacts/models/bootstrap/concept_qa_cifar10_reference.pth


In [5]:
sensitive_labels_dir = paths.sensitive_labels_root

label_files = [
    sensitive_labels_dir / "s_soft_train.npy",
    sensitive_labels_dir / "s_hard_train.npy",
    sensitive_labels_dir / "s_soft_test.npy",
    sensitive_labels_dir / "s_hard_test.npy",
]

if all(path.exists() for path in label_files):
    sensitive_label_cache = load_sensitive_labels(sensitive_labels_dir)
    label_source = "cache"
else:
    s_soft_train, s_hard_train = build_sensitive_labels(
        loader=train_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        sens_idx=sens_idx,
        clip_device=device,
        tau=config.sensitive_tau,
        topk=config.sensitive_topk,
        desc="Building sensitive labels (train)",
    )
    s_soft_test, s_hard_test = build_sensitive_labels(
        loader=test_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        sens_idx=sens_idx,
        clip_device=device,
        tau=config.sensitive_tau,
        topk=config.sensitive_topk,
        desc="Building sensitive labels (test)",
    )
    save_sensitive_labels(
        sensitive_labels_dir,
        train_soft=s_soft_train,
        train_hard=s_hard_train,
        test_soft=s_soft_test,
        test_hard=s_hard_test,
    )
    sensitive_label_cache = load_sensitive_labels(sensitive_labels_dir)
    label_source = "built_and_saved"

print(f"Sensitive labels ready from: {label_source} -> {sensitive_labels_dir}")
print(
    {
        "s_soft_train": sensitive_label_cache["s_soft_train"].shape,
        "s_hard_train": sensitive_label_cache["s_hard_train"].shape,
        "s_soft_test": sensitive_label_cache["s_soft_test"].shape,
        "s_hard_test": sensitive_label_cache["s_hard_test"].shape,
    }
)
print(
    "Hard-positive rate (train/test):",
    float(sensitive_label_cache["s_hard_train"].mean()),
    float(sensitive_label_cache["s_hard_test"].mean()),
)

answer_cache_dir = paths.artifacts_root / "concept_answers" / "cifar10"
answer_cache_metadata = {
    "dataset": "cifar10",
    "concept_count": len(concepts),
    "qa_checkpoint": Path(qa_source).name,
    "qa_checkpoint_sha256": file_sha256(qa_source),
    "threshold": config.threshold_for_binarization,
    "sensitive_target": "soft_concept_match",
}
answer_cache_paths = {
    "train": answer_cache_dir / "train_hard_answers.pt",
    "test": answer_cache_dir / "test_hard_answers.pt",
}


@torch.no_grad()
def build_answer_cache(dataset, sensitive_targets, path):
    loader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=device.type == "cuda",
    )
    answers_parts, labels_parts = [], []
    for images, labels in loader:
        answers_parts.append(
            concept_answers_batch(
                images=images,
                model_clip=model_clip,
                dictionary=dictionary,
                answering_model=answering_model,
                clip_device=device,
                train_device=device,
                threshold=config.threshold_for_binarization,
            ).cpu()
        )
        labels_parts.append(labels.cpu())
    save_answer_cache(
        path,
        answers=torch.cat(answers_parts),
        labels=torch.cat(labels_parts),
        sensitive_targets=torch.as_tensor(sensitive_targets),
        metadata=answer_cache_metadata,
    )


for split, dataset, sensitive_targets in (
    ("train", train_ds, sensitive_label_cache["s_soft_train"]),
    ("test", test_ds, sensitive_label_cache["s_soft_test"]),
):
    path = answer_cache_paths[split]
    if FORCE_REBUILD_ANSWER_CACHE or not path.exists():
        build_answer_cache(dataset, sensitive_targets, path)

train_answer_cache = load_answer_cache(
    answer_cache_paths["train"], expected_metadata=answer_cache_metadata
)
test_answer_cache = load_answer_cache(
    answer_cache_paths["test"], expected_metadata=answer_cache_metadata
)
train_loader = make_cached_answer_loader(
    train_answer_cache,
    batch_size=config.batch_size,
    shuffle=True,
    pin_memory=device.type == "cuda",
)
test_loader = make_cached_answer_loader(
    test_answer_cache,
    batch_size=config.batch_size,
    shuffle=False,
    pin_memory=device.type == "cuda",
)
print({"answer_cache": "ready", "train_rows": len(train_loader.dataset)})

Sensitive labels ready from: cache -> /home/jupyter/claq/artifacts/sensitive_labels/cifar10
{'s_soft_train': (50000,), 's_hard_train': (50000,), 's_soft_test': (10000,), 's_hard_test': (10000,)}
Hard-positive rate (train/test): 0.5468999743461609 0.5491999983787537
{'answer_cache': 'ready', 'train_rows': 50000}


In [6]:
def load_or_train_bundle(
    run_name,
    lambda_s,
    lambda_c,
    seed,
    min_history=config.min_history,
    max_history=config.max_history,
    non_sensitive_only=config.non_sensitive_history_only,
    epochs=2,
    learning_rate=config.learning_rate,
    max_train_batches=60 if device.type == "cpu" else None,
    max_test_batches=30 if device.type == "cpu" else None,
    force_retrain=False,
):
    seed_everything(seed)
    run_stem = f"{experiment_name}_{run_name}_seed_{seed}"
    ckpt_path = paths.checkpoints_root / f"{run_stem}_best.pt"
    history_path = paths.runs_root / f"{run_stem}_history.json"
    if ckpt_path.exists() and history_path.exists() and not force_retrain:
        bundle = load_run_bundle(
            ckpt_path,
            device=device,
            max_queries=config.max_queries,
            num_classes=config.num_classes,
        )
        bundle.update({"run_name": run_name, "lambda_s": lambda_s, "seed": seed})
        return bundle

    actor_checkpoint = str(paths.bootstrap_actor_checkpoint) if paths.bootstrap_actor_checkpoint.exists() else None
    classifier_checkpoint = str(paths.bootstrap_classifier_checkpoint) if paths.bootstrap_classifier_checkpoint.exists() else None

    actor, classifier, s_head = build_claq_models(
        max_queries=config.max_queries,
        num_classes=config.num_classes,
        device=device,
        actor_eps=config.actor_eps,
        actor_checkpoint=actor_checkpoint,
        classifier_checkpoint=classifier_checkpoint,
    )
    optimizer = torch.optim.Adam(
        list(actor.parameters()) + list(classifier.parameters()) + list(s_head.parameters()),
        lr=learning_rate,
    )
    history_config = HistorySamplingConfig(
        min_history=min_history,
        max_history=max_history,
        non_sensitive_only=non_sensitive_only,
    )
    history, best = fit_claq(
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        optimizer=optimizer,
        train_loader=train_loader,
        test_loader=test_loader,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        sens_idx=sens_idx,
        history_config=history_config,
        clip_device=device,
        train_device=device,
        threshold_for_binarization=config.threshold_for_binarization,
        lambda_s=lambda_s,
        lambda_c=lambda_c,
        sensitive_tau=config.sensitive_tau,
        sensitive_topk=config.sensitive_topk,
        num_epochs=epochs,
        max_train_batches=max_train_batches,
        max_test_batches=max_test_batches,
    )
    actor.load_state_dict(best["actor_state_dict"])
    classifier.load_state_dict(best["classifier_state_dict"])
    s_head.load_state_dict(best["s_head_state_dict"])
    save_bundle_checkpoint(
        checkpoint_path=ckpt_path,
        actor=actor,
        classifier=classifier,
        s_head=s_head,
        metadata={
            "run_name": run_name,
            "lambda_s": lambda_s,
            "lambda_c": lambda_c,
            "seed": seed,
            "sensitive_conditioning": "conditional_y",
            "best_test_acc": best["test_acc"],
            "best_epoch": best["epoch"],
            "history_config": {
                "min_history": history_config.min_history,
                "max_history": history_config.max_history,
                "non_sensitive_only": history_config.non_sensitive_only,
            },
        },
    )
    history_with_seed = [
        {**row, "run_name": run_name, "lambda_s": lambda_s, "seed": seed}
        for row in history
    ]
    with open(history_path, "w", encoding="utf-8") as handle:
        json.dump(history_with_seed, handle, indent=2)
    bundle = load_run_bundle(
        ckpt_path,
        device=device,
        max_queries=config.max_queries,
        num_classes=config.num_classes,
    )
    bundle.update({"run_name": run_name, "lambda_s": lambda_s, "seed": seed})
    return bundle


primary_bundles_by_seed = {
    seed: {
        "baseline": load_or_train_bundle(
            "baseline", lambda_s=0.0, lambda_c=0.0, seed=seed, epochs=5
        ),
        "claq": load_or_train_bundle(
            "lam_0.40", lambda_s=0.4, lambda_c=0.0, seed=seed, epochs=5
        ),
    }
    for seed in SEEDS
}
baseline_bundle = primary_bundles_by_seed[QUALITATIVE_SEED]["baseline"]
claq_bundle = primary_bundles_by_seed[QUALITATIVE_SEED]["claq"]

print(f"Qualitative seed: {QUALITATIVE_SEED}")
print(baseline_bundle["ckpt_path"])
print(claq_bundle["ckpt_path"])

def answer_builder(images):
    return concept_answers_batch(
        images=images,
        model_clip=model_clip,
        dictionary=dictionary,
        answering_model=answering_model,
        clip_device=device,
        train_device=device,
        threshold=config.threshold_for_binarization,
    )

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

...

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

CLAQ epochs:   0%|          | 0/5 [00:00<?, ?it/s]

Qualitative seed: 0
/home/jupyter/claq/artifacts/models/cifar10_conditional_y_baseline_seed_0_best.pt
/home/jupyter/claq/artifacts/models/cifar10_conditional_y_lam_0.40_seed_0_best.pt


In [7]:
intuition_records = sample_intuition_replays(
    dataset=test_ds,
    answer_builder=answer_builder,
    baseline_bundle=baseline_bundle,
    claq_bundle=claq_bundle,
    concepts=concepts,
    sensitive_mask=sensitive_mask,
    class_names=raw_test_ds.classes,
    num_cases=10,
    pool_size=400 if device.type == "cpu" else 1500,
    prefer_baseline_sensitive=True,
    balance_labels=True,
    random_seed=QUALITATIVE_SEED,
)

intuition_fig = plot_rollout_comparisons(
    records=intuition_records,
    raw_dataset=raw_test_ds,
    output_path=figure_path("cifar10_intuition_replay_examples"),
    title_prefix="tiny-start replay",
)

print(f"Saved tiny-start replay figure: {intuition_fig}")

Sampling intuition replays:   0%|          | 0/1500 [00:00<?, ?it/s]

Saved tiny-start replay figure: /home/jupyter/claq/artifacts/figures/cifar10_intuition_replay_examples.svg
